# Fase B — Limpieza y Normalización
Pipeline: diagnóstico, nulos, outliers, encoding, transformación y guardado en `datos_limpios.parquet`.

In [ ]:
# ZANAHORIA_2026 - Fase B, paso 1: imports, carga y diagnóstico inicial
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import OrdinalEncoder
from scipy.stats import skew

CSV_PATH = "MEN_ESTADISTICAS_EN_EDUCACION_EN_PREESCOLAR,_BÁSICA_Y_MEDIA_POR_MUNICIPIO_20261014.csv"
zanahoria_2026_df_raw = pd.read_csv(CSV_PATH, sep=None, engine="python")

print("Filas originales:", zanahoria_2026_df_raw.shape[0])
print("Duplicados exactos:", zanahoria_2026_df_raw.duplicated().sum())
print("Nulos por columna:")
print(zanahoria_2026_df_raw.isna().sum())

In [ ]:
# ZANAHORIA_2026 - Fase B, paso 2: funciones auxiliares
def ZANAHORIA_2026_a_numero(serie):
    """
    ZANAHORIA_2026
    Convierte texto con porcentajes (ej. '56.11%', '25,6%') a número float.
    Lo que no se pueda convertir queda como NaN.
    """
    limpia = (serie.astype(str)
              .str.replace("%", "", regex=False)
              .str.replace(",", ".", regex=False)
              .str.strip())
    return pd.to_numeric(limpia, errors="coerce")


def ZANAHORIA_2026_outliers_iqr(serie):
    """
    ZANAHORIA_2026
    Marca los valores atípicos de una columna con el método IQR.
    Devuelve True para los atípicos y False para el resto.
    """
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    return (serie < limite_inferior) | (serie > limite_superior)

In [ ]:
# ZANAHORIA_2026 - Fase B, paso 3: copia de trabajo, quitar NACIONAL y duplicados
zanahoria_2026_df_limpio = zanahoria_2026_df_raw.copy()

# Las filas NACIONAL son totales, no municipios
es_nacional = zanahoria_2026_df_limpio["MUNICIPIO"].str.upper().str.strip() == "NACIONAL"
zanahoria_2026_df_limpio = zanahoria_2026_df_limpio[~es_nacional]
print("Filas NACIONAL quitadas:", es_nacional.sum())

# ZANAHORIA_2026: quitar duplicados exactos
zanahoria_2026_df_limpio = zanahoria_2026_df_limpio.drop_duplicates().reset_index(drop=True)
print("Filas después de quitar duplicados:", zanahoria_2026_df_limpio.shape[0])

In [ ]:
# ZANAHORIA_2026 - Fase B, paso 4: unificar nombres usando el código
# (así "Bogotá D.C." y "Bogotá, D.C." quedan iguales)
variantes_muni = zanahoria_2026_df_limpio.groupby("CÓDIGO_MUNICIPIO")["MUNICIPIO"].nunique()
print("Municipios con más de un nombre:", (variantes_muni > 1).sum())

# Se toma el nombre más frecuente de cada código
nombre_muni = (zanahoria_2026_df_limpio.groupby("CÓDIGO_MUNICIPIO")["MUNICIPIO"]
               .agg(lambda s: s.value_counts().index[0]))
zanahoria_2026_df_limpio["MUNICIPIO"] = zanahoria_2026_df_limpio["CÓDIGO_MUNICIPIO"].map(nombre_muni)

# ZANAHORIA_2026: lo mismo para departamentos
nombre_depto = (zanahoria_2026_df_limpio.groupby("CÓDIGO_DEPARTAMENTO")["DEPARTAMENTO"]
                .agg(lambda s: s.value_counts().index[0]))
zanahoria_2026_df_limpio["DEPARTAMENTO"] = zanahoria_2026_df_limpio["CÓDIGO_DEPARTAMENTO"].map(nombre_depto)

# Revisar que Bogotá quedó con un solo nombre (si su código no es 11001, cambiarlo)
print(zanahoria_2026_df_limpio.loc[zanahoria_2026_df_limpio["CÓDIGO_MUNICIPIO"] == 11001, "MUNICIPIO"].unique())

In [ ]:
# ZANAHORIA_2026 - Fase B, paso 5: quitar columnas y convertir porcentajes a número
columnas_quitar = (
    # Aprobación y reprobación suman casi 100% con la deserción (data leakage)
    [c for c in zanahoria_2026_df_limpio.columns if c.startswith("APROBACIÓN") or c.startswith("REPROBACIÓN")]
    # Deserción por nivel son partes del objetivo
    + ["DESERCIÓN_TRANSICIÓN", "DESERCIÓN_PRIMARIA", "DESERCIÓN_SECUNDARIA", "DESERCIÓN_MEDIA"]
    # ~50% nulos (solo 2011-2017) y códigos de ETC sin valor predictivo
    + ["TAMAÑO_PROMEDIO_DE_GRUPO", "SEDES_CONECTADAS_A_INTERNET", "ETC", "CÓDIGO_ETC"]
)
zanahoria_2026_df_limpio = zanahoria_2026_df_limpio.drop(columns=columnas_quitar)
print("Columnas que quedan:", zanahoria_2026_df_limpio.shape[1])

# ZANAHORIA_2026: convertir las columnas de porcentaje a número
columnas_texto = ["MUNICIPIO", "DEPARTAMENTO"]
columnas_id = ["AÑO", "CÓDIGO_MUNICIPIO", "CÓDIGO_DEPARTAMENTO"]
columnas_numericas = [c for c in zanahoria_2026_df_limpio.columns if c not in columnas_texto + columnas_id]
for col in columnas_numericas:
    zanahoria_2026_df_limpio[col] = ZANAHORIA_2026_a_numero(zanahoria_2026_df_limpio[col])

# Sin deserción no se puede entrenar (es el objetivo), se quitan esas filas
zanahoria_2026_df_limpio = zanahoria_2026_df_limpio.dropna(subset=["DESERCIÓN"]).reset_index(drop=True)
print("Filas con deserción válida:", zanahoria_2026_df_limpio.shape[0])

In [ ]:
# ZANAHORIA_2026 - Fase B, paso 6: manejo de nulos con dos estrategias
# Estrategia 1: KNN para coberturas y tasa (variables relacionadas entre sí)
columnas_knn = ["COBERTURA_NETA", "COBERTURA_BRUTA", "TASA_MATRICULACIÓN_5_16"]
columnas_vecinos = ["AÑO", "COBERTURA_NETA", "COBERTURA_BRUTA", "TASA_MATRICULACIÓN_5_16"]

imputador = KNNImputer(n_neighbors=5)
matriz_vecinos = imputador.fit_transform(zanahoria_2026_df_limpio[columnas_vecinos])
zanahoria_2026_vecinos_df = pd.DataFrame(matriz_vecinos, columns=columnas_vecinos,
                                         index=zanahoria_2026_df_limpio.index)

# ZANAHORIA_2026: solo se llenan los nulos de las columnas KNN
for col in columnas_knn:
    zanahoria_2026_df_limpio[col] = zanahoria_2026_df_limpio[col].fillna(zanahoria_2026_vecinos_df[col])

# Estrategia 2: mediana para el resto de columnas numéricas con nulos
columnas_mediana = [c for c in columnas_numericas
                    if c in zanahoria_2026_df_limpio.columns
                    and c not in columnas_knn
                    and zanahoria_2026_df_limpio[c].isna().any()]
for col in columnas_mediana:
    zanahoria_2026_df_limpio[col] = zanahoria_2026_df_limpio[col].fillna(zanahoria_2026_df_limpio[col].median())

print("Nulos que quedan:", zanahoria_2026_df_limpio.isna().sum().sum())

**Justificación de la imputación**
- **KNN** en cobertura y tasa: cambian juntas, así que el promedio de municipios parecidos es más realista que la mediana.
- **Mediana** en el resto: resiste valores extremos (la media no) y las variables tienen colas largas.
- No se usa `fillna(0)` (como en el P1): un cero significa "no hay estudiantes" y distorsiona el modelo.

In [ ]:
# ZANAHORIA_2026 - Fase B, paso 7: detectar outliers con IQR
columnas_outliers = ["POBLACIÓN_5_16", "TASA_MATRICULACIÓN_5_16", "COBERTURA_NETA",
                     "COBERTURA_BRUTA", "REPITENCIA", "DESERCIÓN"]

for col in columnas_outliers:
    marcas = ZANAHORIA_2026_outliers_iqr(zanahoria_2026_df_limpio[col])
    print(f"{col}: {marcas.sum()} atípicos ({marcas.mean() * 100:.1f}%)")

**Decisión sobre outliers: SE CONSERVAN.**
Las coberturas mayores a 100% son reales (incluyen estudiantes con edad mayor a la esperada). Eliminarlos quitaría justo los municipios con rezago escolar, que son el tema del análisis. Para la población no se eliminan: se aplica transformación logarítmica.

In [ ]:
# ZANAHORIA_2026 - Fase B, paso 8: transformación logarítmica de la población (muy sesgada)
print("Asimetría antes:", round(skew(zanahoria_2026_df_limpio["POBLACIÓN_5_16"]), 2))

# log1p = log(1 + x), funciona aunque haya ceros
zanahoria_2026_df_limpio["LOG_POBLACIÓN_5_16"] = np.log1p(zanahoria_2026_df_limpio["POBLACIÓN_5_16"])

print("Asimetría después:", round(skew(zanahoria_2026_df_limpio["LOG_POBLACIÓN_5_16"]), 2))

In [ ]:
# ZANAHORIA_2026 - Fase B, paso 9: Ordinal Encoding (Pequeño < Mediano < Grande)
zanahoria_2026_df_limpio["CATEGORÍA_POBLACIÓN"] = pd.qcut(
    zanahoria_2026_df_limpio["POBLACIÓN_5_16"].rank(method="first"),
    3,
    labels=["Pequeño", "Mediano", "Grande"]
).astype(str)

# El orden se indica de forma explícita
codificador_ordinal = OrdinalEncoder(categories=[["Pequeño", "Mediano", "Grande"]])
zanahoria_2026_df_limpio["CATEGORÍA_POBLACIÓN_ORD"] = (
    codificador_ordinal.fit_transform(zanahoria_2026_df_limpio[["CATEGORÍA_POBLACIÓN"]]).astype(int)
)

# ZANAHORIA_2026: se quita el texto y la población original (ya existen LOG y ORD)
zanahoria_2026_df_limpio = zanahoria_2026_df_limpio.drop(columns=["CATEGORÍA_POBLACIÓN", "POBLACIÓN_5_16"])
print(zanahoria_2026_df_limpio["CATEGORÍA_POBLACIÓN_ORD"].value_counts())

In [ ]:
# ZANAHORIA_2026 - Fase B, paso 10: One-Hot Encoding del departamento (nominal, sin orden)
zanahoria_2026_df_limpio = pd.get_dummies(zanahoria_2026_df_limpio,
                                          columns=["DEPARTAMENTO"],
                                          drop_first=True,
                                          dtype=int)
print("Columnas después del One-Hot:", zanahoria_2026_df_limpio.shape[1])

In [ ]:
# ZANAHORIA_2026 - Fase B, paso 11: revisar asimetría para elegir el escalador
# La normalización se aplica DESPUÉS del train_test_split (en fase_c) para evitar data leakage.
excluir = ["DESERCIÓN", "AÑO", "CÓDIGO_MUNICIPIO", "CÓDIGO_DEPARTAMENTO", "CATEGORÍA_POBLACIÓN_ORD"]
columnas_escalar = [c for c in zanahoria_2026_df_limpio.select_dtypes("number").columns
                    if c not in excluir and not c.startswith("DEPARTAMENTO_")]

asimetria = zanahoria_2026_df_limpio[columnas_escalar].apply(skew).round(2)
print(asimetria.sort_values())

**Regla de escalado (se aplica en fase_c, después del split)**
- Asimetría absoluta > 1 → RobustScaler (usa mediana y rango intercuartílico).
- En otro caso → StandardScaler (media 0, desviación 1).
- Las columnas one-hot (0/1) no se escalan.

In [ ]:
# ZANAHORIA_2026 - Fase B, paso 12: guardar el DataFrame limpio
zanahoria_2026_df_limpio.to_parquet("datos_limpios.parquet", index=False)
print("Guardado datos_limpios.parquet con forma:", zanahoria_2026_df_limpio.shape)